# Notebook 02 — Text Processing

**Project:** Iqraa AI — Quranic recitation correction for Riwayat Qaloon an Nafi  
**Purpose:** Take `qalon_canonical.json` and produce a clean, model-ready
vocabulary and normalised text dataset.

This notebook does three things:

1. **Normalise** every ayah — strip KFGQPC structural marks and ayah-number
   glyphs while keeping all tashkeel (diacritics) intact
2. **Lock away a protected test set** — 22 ayahs that are never seen during
   training and used only for the final evaluation
3. **Build the character vocabulary** — the exhaustive list of every character
   the model is allowed to predict, with a stable integer index for each one

The outputs of this notebook are consumed directly by Notebook 04 (training).

**Safe to re-run:** existing splits are detected and reused automatically.

## Cell 2 — Mount Google Drive and Configure All Paths

We mount Drive and define every path constant in one place.
`SPLITS_DIR` is new in this notebook — it stores the three dataset splits
(protected test, train, validation) so they survive Colab session restarts
and remain identical across all subsequent notebooks.

`PROCESSED_DIR` holds model-ready artefacts such as `vocab.json` that
the training notebook reads directly.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE    = '/content/drive/MyDrive/IqraaAI_Dataset'
AUDIO_DIR     = DRIVE_BASE + '/audio/hudhaifi_qaloon'
TEXT_FILE     = DRIVE_BASE + '/text/qalon_canonical.json'
OUTPUT_DIR    = DRIVE_BASE + '/processed'
SPLITS_DIR    = DRIVE_BASE + '/splits'
PROCESSED_DIR = DRIVE_BASE + '/processed'

import os
os.makedirs(SPLITS_DIR,    exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

print('Drive mounted.')
print(f'Text file     : {TEXT_FILE}')
print(f'Splits dir    : {SPLITS_DIR}')
print(f'Processed dir : {PROCESSED_DIR}')

## Cell 3 — What "Normalisation" Means for KFGQPC Arabic Text

The `qalon_canonical.json` file uses the **KFGQPC (King Fahd Glorious Quran
Printing Complex) digital font standard**. This standard introduces Unicode
characters beyond the standard Arabic tashkeel in order to encode Quranic
recitation annotations — symbols that appear visually in the printed mushaf
but are **not phonemic**.

### What needs to be removed

**KFGQPC structural marks (U+06D6–U+06ED):**  
Characters like `اِ۬` contain a combining mark (the `۬` glyph) that tells the
typesetting engine to render the letter with a special visual form — a small
circle, a dotless head, a high stop. These marks carry **zero phonemic
information**. The model does not need to predict them and must not see them
in the label sequence, otherwise the vocabulary explodes with symbols the
audio signal cannot distinguish.

**Arabic-Indic digit characters (U+0660–U+0669):**  
Each ayah value in the JSON ends with the ayah number written in Arabic-Indic
numerals (e.g. `١`, `٢`, `٣`). These numbers are visual markers in the
printed mushaf, not recited syllables. They must be stripped.

### What must be kept

**All tashkeel (U+064B–U+0652, U+0670):**  
Fatḥa (َ), kasra (ِ), ḍamma (ُ), sukūn (ْ), shadda (ّ), and the six forms
of tanwīn are the **entire point of the system**. Qaloon differs from Hafs
in which diacritics appear on specific words. The model must learn to predict
the correct diacritic on each base letter — stripping them would make the
problem trivially wrong.

## Cell 4 — Load qalon_canonical.json and Check for Existing Splits

We always load the full 6 214-ayah dict. We then check whether the three
split files already exist on Drive from a previous run of this notebook.

If splits exist, later cells will reload them rather than recreating them.
This is important for the **protected test set** — its membership must never
change between runs, otherwise evaluation results become incomparable.

In [ ]:
import json

with open(TEXT_FILE, encoding='utf-8') as f:
    quran = json.load(f)

print(f'Loaded {len(quran)} ayahs from qalon_canonical.json')
print(f'Key format sample: {list(quran.keys())[:3]}')
print()

protected_test_path = SPLITS_DIR + '/protected_test.json'
train_text_path     = SPLITS_DIR + '/train_text.json'
val_text_path       = SPLITS_DIR + '/val_text.json'

if os.path.exists(protected_test_path) and os.path.exists(train_text_path):
    with open(protected_test_path, encoding='utf-8') as f:
        _existing_test  = json.load(f)
    with open(train_text_path, encoding='utf-8') as f:
        _existing_train = json.load(f)
    print('Using existing splits:')
    print(f'  Protected test : {len(_existing_test)} ayahs')
    print(f'  Train split    : {len(_existing_train)} ayahs')
    if os.path.exists(val_text_path):
        with open(val_text_path, encoding='utf-8') as f:
            _existing_val = json.load(f)
        print(f'  Val split      : {len(_existing_val)} ayahs')
else:
    print('No existing splits found — will create new splits in later cells.')

## Cell 5 — The Five Normalisation Steps and Why Each Exists

Our normalisation pipeline applies five transformations in order:

**Step 1 — NFC Unicode normalisation:**  
Unicode allows the same character to be represented in multiple ways
(e.g. a letter with a diacritic can be stored as one precomposed codepoint
or as two separate codepoints). NFC (Canonical Decomposition followed by
Canonical Composition) collapses all representations to a single canonical
form. Without this, two visually identical ayahs could have different byte
sequences and the vocabulary would contain duplicate tokens.

**Step 2 — Remove KFGQPC structural marks (U+06D6–U+06ED):**  
The visual annotation marks described in Cell 3. Phonemically empty.

**Step 3 — Remove Arabic-Indic digit characters (U+0660–U+0669):**  
Ayah-number glyphs appended to each value in the JSON. Not recited.

**Step 4 — Remove KFGQPC letter mark U+0620 and non-breaking space U+00A0:**  
U+0620 (Arabic Letter Kashmiri Ya) is used in KFGQPC as a special placeholder
glyph. U+00A0 (non-breaking space) appears at the end of each JSON value
before the ayah numeral. Both are artefacts of the typesetting format.

**Step 5 — Collapse whitespace and strip edges:**  
After removals, gaps may be left in the middle of the string. We collapse
consecutive spaces to one and trim leading/trailing whitespace.

**What we never touch:**  
All standard Arabic letters (U+0621–U+063A, U+0641–U+064A) and all tashkeel
(U+064B–U+0652, U+0670) pass through unchanged.

## Cell 6 — Implement and Test normalize_qalon_text

We define the normalisation function and run it on five representative ayahs
— Al-Fatihah (Ayah 1), Al-Baqarah (Ayah 1), Ayat al-Kursi, Al-Ikhlas
(Ayah 1), and An-Nas (Ayah 1) — to verify the before/after transformation
looks correct before we apply it to all 6 214 ayahs.

In [ ]:
import re
import unicodedata

# Build the removal set from explicit Unicode codepoint ranges.
# Using chr() avoids escape-sequence confusion across editors and systems.
_REMOVE_CHARS = set(
    [chr(c) for c in range(0x06D6, 0x06DD)]   # U+06D6..U+06DC  Quranic marks
    + [chr(c) for c in range(0x06DF, 0x06E5)] # U+06DF..U+06E4  more marks
    + [chr(0x06E7), chr(0x06E8)]               # small high yeh / noon
    + [chr(c) for c in range(0x06EA, 0x06EE)] # U+06EA..U+06ED  low stops
    + [chr(c) for c in range(0x0660, 0x066A)] # U+0660..U+0669  Arabic-Indic digits
    + [chr(0x0620), chr(0x00A0)]               # KFGQPC mark, non-breaking space
)


def normalize_qalon_text(text: str) -> str:
    """Normalise a single Qaloon ayah for use as a CTC training label.

    Removes KFGQPC structural marks, Arabic-Indic digit characters, and
    non-breaking spaces. Preserves all tashkeel (diacritics). Returns a
    clean Arabic string with single spaces between words.
    """
    # Step 1: canonical Unicode composition
    text = unicodedata.normalize('NFC', text)
    # Steps 2–4: remove structural marks, digits, KFGQPC artefacts
    text = ''.join(ch for ch in text if ch not in _REMOVE_CHARS)
    # Step 5: collapse whitespace and strip edges
    text = re.sub(r'\s+', ' ', text).strip()
    return text


# Test on 5 representative ayahs
SAMPLE_KEYS = ['001001', '002001', '002255', '112001', '114001']

print('Normalisation samples (before → after):')
print('=' * 60)
for key in SAMPLE_KEYS:
    if key not in quran:
        print(f'  Key {key} not found — skipping')
        continue
    before = quran[key]
    after  = normalize_qalon_text(before)
    surah  = int(key[:3])
    ayah   = int(key[3:])
    print(f'\n  [{key}] Surah {surah}, Ayah {ayah}')
    print(f'  Before : {before}')
    print(f'  After  : {after}')
    chars_removed = len(before) - len(after)
    print(f'  Chars removed: {chars_removed}')

## Cell 7 — The Protected Test Set: Your Real Exam

Imagine you are studying for an exam. Your teacher gives you a set of
practice questions to study from. On exam day, the same questions appear
word-for-word. You score 100% — but not because you understood the material.
You scored 100% because you memorised the answers. You have no idea how you
would perform on questions you had never seen before.

**This is the exact problem we must prevent in machine learning.**

If the model is trained on an ayah and then evaluated on that same ayah,
its score will be artificially inflated. It has "seen the exam". The reported
Word Error Rate will look better than it truly is, and we will not know whether
the model has actually learned to recognise Qaloon recitation or merely
memorised the training examples.

**The solution: lock 22 ayahs away before training begins.**

We choose 22 ayahs that cover a wide range of surahs, lengths, and phonological
features — including the frequently-recited Al-Fatihah, Ayat al-Kursi, Al-Ikhlas,
and one ayah from each of ten shorter surahs. These 22 ayahs are:

- Written to disk **right now**, before any training happens
- **Never used** to compute training loss or validation metrics
- Used **only once**, in Notebook 05, to report the final evaluation numbers

The specific 22 keys below are **fixed for the lifetime of this project**.
Do not change them.

In [ ]:
# These 22 keys are fixed — do not change them.
PROTECTED_KEYS = [
    # Al-Fatihah in full (7 ayahs)
    '001001', '001002', '001003', '001004', '001005', '001006', '001007',
    # Ayat al-Kursi
    '002255',
    # Al-Ikhlas in full (4 ayahs)
    '112001', '112002', '112003', '112004',
    # One ayah each from 10 shorter surahs
    '078001', '080001', '087001', '091007', '092005',
    '093006', '097003', '099001', '101004', '103002',
]

protected_test_path = SPLITS_DIR + '/protected_test.json'

if os.path.exists(protected_test_path):
    with open(protected_test_path, encoding='utf-8') as f:
        protected_test = json.load(f)
    # Re-normalise in case the function was updated
    protected_test = {k: normalize_qalon_text(v) for k, v in protected_test.items()}
    print(f'Loaded existing protected test set ({len(protected_test)} ayahs)')
else:
    protected_test = {}
    for key in PROTECTED_KEYS:
        if key in quran:
            protected_test[key] = normalize_qalon_text(quran[key])
        else:
            print(f'  WARNING: key {key} not found in quran dict')
    with open(protected_test_path, 'w', encoding='utf-8') as f:
        json.dump(protected_test, f, ensure_ascii=False, indent=2)
    print(f'Created protected test set ({len(protected_test)} ayahs)')

print(f'\nAll {len(protected_test)} protected ayahs:')
print('-' * 55)
for key, text in protected_test.items():
    surah = int(key[:3])
    ayah  = int(key[3:])
    print(f'  [{key}] Surah {surah:>3}, Ayah {ayah:>3} : {text}')

## Cell 9 — The Vocabulary: What the Model Is Allowed to Predict

Wav2Vec2ForCTC is a **sequence-to-sequence model** that reads raw audio frames
and outputs a probability distribution over a fixed vocabulary at every frame.
The CTC decoder then collapses the frame-by-frame probabilities into a final
character sequence.

This means the model can **only predict characters that appear in the
vocabulary**. If a diacritic is missing from the vocabulary, the model can
never output it — not even if that diacritic is the correct answer.

### Why character-level?

Arabic tashkeel attaches to base letters. The word `كَتَبَ` (kataba, he wrote)
contains three base letters and three fatha diacritics. A word-level vocabulary
would need a separate entry for every possible (base letter + diacritic)
combination — tens of thousands of entries. A **character-level** vocabulary
reduces this to ~80 tokens (28 Arabic letters + ~15 diacritic marks + specials)
while still allowing the model to reconstruct any diacritized Arabic word.

### Special tokens

Three tokens are not Arabic characters but are required by the CTC framework:

| Token | Index | Role |
|-------|-------|------|
| `[PAD]` | 0 | **CTC blank token.** Must be index 0. The CTC decoder uses this to separate repeated characters and to fill frames where no character is predicted. |
| `\|` | N−1 | **Word delimiter.** Represents a space between words. Wav2Vec2 convention: spaces in the text label become `\|` in the token sequence. |
| `[UNK]` | N | **Unknown token.** Any character not in the vocabulary maps here. Placed last to make it easy to spot in debugging. |

In [ ]:
from collections import Counter

# Build vocabulary from all training ayahs (full corpus minus protected test).
# We use ALL non-test ayahs here, not just the 90% train split, so the vocab
# covers characters that appear only in the validation portion as well.
training_texts = [
    normalize_qalon_text(text)
    for key, text in quran.items()
    if key not in protected_test
]
print(f'Non-test ayahs used for vocab: {len(training_texts)}')

char_counts: Counter = Counter()
for text in training_texts:
    for ch in text:
        if ch != ' ':   # spaces become the | token, counted separately
            char_counts[ch] += 1

print(f'Unique characters found: {len(char_counts)}')
print()
print(f'  {"Codepoint":<12} {"Char":<8} {"Count":<10} Unicode Name')
print('  ' + '-' * 58)
for ch, count in char_counts.most_common():
    try:
        name = unicodedata.name(ch)
    except ValueError:
        name = '(no name)'
    print(f'  U+{ord(ch):04X}      {repr(ch):<8} {count:<10} {name}')

# Build vocab: [PAD]=0, chars sorted by descending frequency, then |, then [UNK]
vocab = {'[PAD]': 0}
for i, (ch, _) in enumerate(char_counts.most_common(), start=1):
    vocab[ch] = i
next_idx    = len(vocab)
vocab['|']      = next_idx
vocab['[UNK]']  = next_idx + 1

vocab_path = PROCESSED_DIR + '/vocab.json'
with open(vocab_path, 'w', encoding='utf-8') as f:
    json.dump(vocab, f, ensure_ascii=False, indent=2)

print()
print(f'Vocabulary size : {len(vocab)} tokens')
print(f'  [PAD]  → index {vocab["[PAD]"]}')
print(f'  |      → index {vocab["|"]}')
print(f'  [UNK]  → index {vocab["[UNK]"]}')
print(f'Saved to: {vocab_path}')

## Cell 11 — The Train / Validation Split

After removing the 22 protected test ayahs, we have 6 192 ayahs left.
We divide them into two sets:

**Training set (90% ≈ 5 572 ayahs):**  
The ayahs the model actually learns from. The training loss is computed on
these and the model weights are updated to reduce it.

**Validation set (10% ≈ 620 ayahs):**  
Ayahs the model never trains on, but that we check during training to monitor
progress. Every N steps the trainer runs inference on the validation set and
reports the Word Error Rate. This lets us:

- Track whether the model is improving or overfitting
- Select the best checkpoint (lowest validation WER) with `load_best_model_at_end`
- Trigger early stopping if validation WER stops improving

**Why not use the test set for this?**  
If we check the test set every N steps during training to decide when to stop,
we are indirectly "leaking" information about the test set into the training
process. The validation set exists precisely to be consumed in this way safely.

**Why seed 42?**  
Random shuffling with a fixed seed ensures the split is reproducible.
Running this cell twice will produce identical train/val assignments.

In [ ]:
import random

# All ayahs except the 22 locked away in the protected test set
remaining_keys = [k for k in quran.keys() if k not in protected_test]
print(f'Total ayahs available for train/val: {len(remaining_keys)}')

# Reproducible shuffle
random.seed(42)
random.shuffle(remaining_keys)

n_train    = int(len(remaining_keys) * 0.9)
train_keys = remaining_keys[:n_train]
val_keys   = remaining_keys[n_train:]

train_data = {k: normalize_qalon_text(quran[k]) for k in train_keys}
val_data   = {k: normalize_qalon_text(quran[k]) for k in val_keys}

train_path = SPLITS_DIR + '/train_text.json'
val_path   = SPLITS_DIR + '/val_text.json'

with open(train_path, 'w', encoding='utf-8') as f:
    json.dump(train_data, f, ensure_ascii=False, indent=2)
with open(val_path, 'w', encoding='utf-8') as f:
    json.dump(val_data, f, ensure_ascii=False, indent=2)

total = len(remaining_keys)
print(f'Train split : {len(train_data):>5} ayahs  ({len(train_data)/total*100:.1f}%)')
print(f'Val split   : {len(val_data):>5} ayahs  ({len(val_data)/total*100:.1f}%)')
print(f'Test (locked): {len(protected_test):>4} ayahs  (held out)')
print()
print(f'Saved train split to : {train_path}')
print(f'Saved val split to   : {val_path}')

## Cell 13 — Summary and What Notebook 03 Will Do

### What this notebook produced

| Output | Path | Description |
|--------|------|-------------|
| Character vocabulary | `processed/vocab.json` | ~80 tokens: `[PAD]`=0, chars by freq, `\|`, `[UNK]` |
| Protected test set | `splits/protected_test.json` | 22 locked ayahs — never train on these |
| Training text | `splits/train_text.json` | ~90% of non-test ayahs, normalised |
| Validation text | `splits/val_text.json` | ~10% of non-test ayahs, normalised |
| Completion marker | `processed/notebook_02_complete.json` | Summary with counts |

### What Notebook 03 (Alignment) will do

We now have 114 surah-level WAV files (from Notebook 01) and a pairing
manifest linking each WAV to its ayahs. We also have the normalised text
labels (from this notebook).

**Notebook 03** uses these to produce the fine-grained alignment:

1. For each surah WAV, split it into individual **ayah-level segments**
   using energy-based silence detection (with Notebook 04 refining this
   with forced alignment)
2. Match each segment to its normalised text label from `train_text.json`
   or `val_text.json`
3. Build a HuggingFace `DatasetDict` with `audio` and `text` columns
4. Validate segment durations and discard outliers

The `DatasetDict` from Notebook 03 is the direct input to Notebook 04 training.

In [ ]:
import json

summary = {
    'notebook'   : '02_text_processing',
    'status'     : 'complete',
    'vocab_size' : len(vocab),
    'train_ayahs': len(train_data),
    'val_ayahs'  : len(val_data),
    'test_ayahs' : 22,
}

marker_path = PROCESSED_DIR + '/notebook_02_complete.json'
with open(marker_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2)

print('Notebook 02 complete.')
print(json.dumps(summary, indent=2))
print(f'\nCompletion marker saved to: {marker_path}')